# Vehicle Detection and Dual-Lane Counting System
## Using YOLOv8n and ByteTrack - SEPARATE LEFT AND RIGHT LANE COUNTS

This notebook implements a complete vehicle detection and counting system that:
- Detects vehicles (motorcycles, bicycles, cars, trucks) using YOLOv8n
- Tracks vehicles using ByteTrack to avoid duplicate counting
- Counts vehicles separately in LEFT and RIGHT lanes
- Outputs a video with annotated detections and dual-lane counts

### Step 1: Install Dependencies

In [ ]:
import subprocess
import sys

packages = ['ultralytics', 'opencv-python', 'numpy']

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

print("All dependencies installed successfully!")

### Step 2: Import Required Libraries

In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
from pathlib import Path
from collections import defaultdict
import warnings

warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


### Step 3: Configure Paths and Parameters

In [25]:
# Define project paths
PROJECT_ROOT = Path("..")
VIDEO_INPUT = PROJECT_ROOT / "data" / "video" / "raw" / "traffic_video_demo.mp4"
VIDEO_OUTPUT = PROJECT_ROOT / "data" / "video" / "processed" / "output_lane_count.mp4"
MODELS_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODELS_DIR / "yolov8m.pt"

# Ensure output directory exists
VIDEO_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Configuration parameters
CONFIG = {
    'model_name': 'yolov8m.pt',
    'model_path': str(MODEL_PATH),
    'confidence_threshold': 0.5,
    'tracker': 'bytetrack.yaml',
    'device': 0,
    'target_classes': ['motorcycle', 'bicycle', 'car', 'truck'],
    'video_input': str(VIDEO_INPUT),
    'video_output': str(VIDEO_OUTPUT),
    'display_fps': True,
    'display_roi': True,
    'draw_track_ids': True,
    'left_count_line_y': 350,
    'right_count_line_y': 400,
}

print("Configuration parameters set:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Configuration parameters set:
  model_name: yolov8m.pt
  model_path: ..\models\yolov8m.pt
  confidence_threshold: 0.5
  tracker: bytetrack.yaml
  device: 0
  target_classes: ['motorcycle', 'bicycle', 'car', 'truck']
  video_input: ..\data\video\raw\traffic_video_demo.mp4
  video_output: ..\data\video\processed\output_lane_count.mp4
  display_fps: True
  display_roi: True
  draw_track_ids: True
  left_count_line_y: 350
  right_count_line_y: 400


### Step 4: Define Vehicle Detector Class (DUAL-LANE)

In [26]:
class VehicleDetector:
    """
    Vehicle detector with separate counting for LEFT and RIGHT lanes.
    """
    
    CLASS_NAMES = {
        0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane',
        5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light',
        10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench',
        14: 'cat', 15: 'dog', 16: 'horse', 17: 'sheep', 18: 'cow', 19: 'elephant',
        20: 'bear', 21: 'zebra', 22: 'giraffe', 23: 'backpack', 24: 'umbrella',
        25: 'handbag', 26: 'tie', 27: 'suitcase', 28: 'frisbee', 29: 'skis',
        30: 'snowboard', 31: 'sports ball', 32: 'kite', 33: 'baseball bat',
        34: 'baseball glove', 35: 'skateboard', 36: 'surfboard', 37: 'tennis racket',
        38: 'bottle', 39: 'wine glass', 40: 'cup', 41: 'fork', 42: 'knife',
        43: 'spoon', 44: 'bowl', 45: 'banana', 46: 'apple', 47: 'sandwich',
        48: 'orange', 49: 'broccoli', 50: 'carrot', 51: 'hot dog', 52: 'pizza',
        53: 'donut', 54: 'cake', 55: 'chair', 56: 'couch', 57: 'potted plant',
        58: 'bed', 59: 'dining table', 60: 'toilet', 61: 'tv', 62: 'laptop',
        63: 'mouse', 64: 'remote', 65: 'keyboard', 66: 'microwave', 67: 'oven',
        68: 'toaster', 69: 'sink', 70: 'refrigerator', 71: 'book', 72: 'clock',
        73: 'vase', 74: 'scissors', 75: 'teddy bear', 76: 'hair drier', 77: 'toothbrush'
    }
    
    def __init__(self, config):
        """Initialize detector with dual-lane tracking."""
        self.track_last_positions = {}
        self.config = config
        self.device = config['device']
        self.confidence_threshold = config['confidence_threshold']
        self.target_classes = config['target_classes']
        
        print(f"Loading YOLOv8n model: {config['model_name']}")
        self.model = YOLO(config['model_name'])
        print("Model loaded successfully!")
        
        # Track IDs counted in each lane
        self.counted_tracks_left = set()
        self.counted_tracks_right = set()
        
        # Separate counts for each lane
        self.vehicle_counts_left = defaultdict(int)
        self.vehicle_counts_right = defaultdict(int)
    
    def get_lane_for_bbox(self, bbox, frame_width):
        """Determine which lane(s) the vehicle is in."""
        x1, y1, x2, y2 = bbox
        bbox_center_x = (x1 + x2) / 2
        frame_mid = frame_width / 2
        
        return {
            'left': bbox_center_x < frame_mid,
            'right': bbox_center_x > frame_mid
        }
    
    def get_vehicle_class(self, class_id):
        """Get class name and check if it's a target class."""
        class_name = self.CLASS_NAMES.get(int(class_id), 'unknown')
        is_target = class_name in self.target_classes
        return class_name, is_target
    
    def process_frame(self, frame):
        """Process frame and track vehicles in both lanes."""
        h, w = frame.shape[:2]
        frame_copy = frame.copy()
        
        # Run YOLOv8 tracking
        results = self.model.track(
            frame,
            persist=True,
            conf=self.confidence_threshold,
            tracker=self.config['tracker'],
            verbose=False
        )
        
        valid_detections = []
        
        if results[0].boxes is not None:
            boxes = results[0].boxes
            for box in boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else None
                
                class_name, is_target = self.get_vehicle_class(cls_id)
                if not is_target:
                    continue
                
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                bbox = [x1, y1, x2, y2]
                lanes = self.get_lane_for_bbox(bbox, w)
                
                # Count in LEFT lane
                # Count in LEFT lane
                if lanes['left'] and track_id is not None:
                    if track_id not in self.counted_tracks_left:
                        self.counted_tracks_left.add(track_id)
                        self.vehicle_counts_left[class_name] += 1

                # Count in RIGHT lane
                if lanes['right'] and track_id is not None:
                    if track_id not in self.counted_tracks_right:
                        self.counted_tracks_right.add(track_id)
                        self.vehicle_counts_right[class_name] += 1

                valid_detections.append({
                    'bbox': bbox,
                    'class_name': class_name,
                    'confidence': conf,
                    'track_id': track_id,
                    'lanes': lanes
                })
        
        frame_annotated = self._draw_detections(frame_copy, valid_detections, w, h)
        return frame_annotated, valid_detections
    
    def _draw_detections(self, frame, detections, frame_width, frame_height):
        """Draw detections with lane indicators."""
        if self.config['display_roi']:
            mid_x = frame_width // 2
            cv2.line(frame, (mid_x, 0), (mid_x, frame_height), (0, 255, 255), 3)
            cv2.putText(frame, "LEFT", (mid_x // 2 - 30, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 100, 0), 2)
            cv2.putText(frame, "RIGHT", (mid_x + 20, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)
        
        for det in detections:
            x1, y1, x2, y2 = det['bbox']
            class_name = det['class_name']
            conf = det['confidence']
            track_id = det['track_id']
            lanes = det['lanes']
            
            # Color based on lane
            if lanes['left'] and lanes['right']:
                color = (0, 255, 255)  # Cyan
            elif lanes['left']:
                color = (255, 100, 0)  # Blue
            else:
                color = (0, 165, 255)  # Orange
            
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            
            label = f"{class_name.capitalize()} {conf:.2f}"
            if self.config['draw_track_ids'] and track_id is not None:
                label += f" [{track_id}]"
            
            if lanes['left'] and not lanes['right']:
                label += " [L]"
            elif lanes['right'] and not lanes['left']:
                label += " [R]"
            
            label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)[0]
            cv2.rectangle(frame, (x1, y1 - 25), (x1 + label_size[0], y1), color, -1)
            cv2.putText(frame, label, (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        return frame
    
    def draw_counts(self, frame, fps=None):
        """Draw dual-lane counts on frame."""
        h, w = frame.shape[:2]
        
        total_left = sum(self.vehicle_counts_left.values())
        total_right = sum(self.vehicle_counts_right.values())
        total_all = total_left + total_right
        
        # LEFT LANE
        left_info = [f"LEFT LANE: {total_left}", 
                     f"Cars: {self.vehicle_counts_left['car']}",
                     f"Trucks: {self.vehicle_counts_left['truck']}",
                     f"Motorcycles: {self.vehicle_counts_left['motorcycle']}",
                     f"Bicycles: {self.vehicle_counts_left['bicycle']}"]
        
        overlay_l = frame.copy()
        cv2.rectangle(overlay_l, (10, 70), (260, 180), (0, 0, 0), -1)
        cv2.addWeighted(overlay_l, 0.75, frame, 0.25, 0, frame)
        
        for idx, text in enumerate(left_info):
            cv2.putText(frame, text, (20, 95 + idx * 22),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 100, 0), 2)
        
        # RIGHT LANE
        right_info = [f"RIGHT LANE: {total_right}",
                      f"Cars: {self.vehicle_counts_right['car']}",
                      f"Trucks: {self.vehicle_counts_right['truck']}",
                      f"Motorcycles: {self.vehicle_counts_right['motorcycle']}",
                      f"Bicycles: {self.vehicle_counts_right['bicycle']}"]
        
        overlay_r = frame.copy()
        right_panel_x = w - 260
        cv2.rectangle(overlay_r, (right_panel_x, 70), (w - 10, 180), (0, 0, 0), -1)
        cv2.addWeighted(overlay_r, 0.75, frame, 0.25, 0, frame)
        
        for idx, text in enumerate(right_info):
            cv2.putText(frame, text, (right_panel_x + 10, 95 + idx * 22),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 165, 255), 2)
        
        # TOTAL
        total_text = f"TOTAL: {total_all}"
        if fps:
            total_text += f" | FPS: {fps:.1f}"
        
        overlay_t = frame.copy()
        cv2.rectangle(overlay_t, (w//2 - 150, h - 50), (w//2 + 150, h - 10), (0, 0, 0), -1)
        cv2.addWeighted(overlay_t, 0.75, frame, 0.25, 0, frame)
        cv2.putText(frame, total_text, (w//2 - 140, h - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        
        return frame
    
    def reset_counts(self):
        """Reset counts for both lanes."""
        self.counted_tracks_left.clear()
        self.counted_tracks_right.clear()
        self.vehicle_counts_left.clear()
        self.vehicle_counts_right.clear()

print("VehicleDetector class defined!")

VehicleDetector class defined!


### Step 5: Video Processing Function

In [27]:
def process_video(detector, input_path, output_path):
    """Process video with dual-lane vehicle detection."""
    if not Path(input_path).exists():
        print(f"ERROR: Input video not found at {input_path}")
        return
    
    print(f"\nProcessing video: {input_path}")
    
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print("ERROR: Failed to open video")
        return
    
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Resolution: {frame_width}x{frame_height}, FPS: {fps}, Frames: {total_frames}")
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
    
    if not out.isOpened():
        print("ERROR: Failed to initialize video writer")
        cap.release()
        return
    
    detector.reset_counts()
    
    frame_count = 0
    fps_list = []
    prev_time = cv2.getTickCount()
    
    print("Processing frames...")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        curr_time = cv2.getTickCount()
        time_diff = (curr_time - prev_time) / cv2.getTickFrequency()
        current_fps = 1 / time_diff if time_diff > 0 else 0
        fps_list.append(current_fps)
        prev_time = curr_time
        
        annotated_frame, _ = detector.process_frame(frame)
        annotated_frame = detector.draw_counts(annotated_frame, current_fps)
        
        out.write(annotated_frame)
        frame_count += 1
        
        if frame_count % 30 == 0 or frame_count == 1:
            progress = (frame_count / total_frames) * 100
            print(f"Progress: {frame_count}/{total_frames} ({progress:.1f}%)")
    
    cap.release()
    out.release()
    
    print("\n" + "="*60)
    print("FINAL RESULTS")
    print("="*60)
    
    total_left = sum(detector.vehicle_counts_left.values())
    total_right = sum(detector.vehicle_counts_right.values())
    
    print(f"\nLEFT LANE: {total_left} vehicles")
    print(f"  Cars: {detector.vehicle_counts_left['car']}")
    print(f"  Trucks: {detector.vehicle_counts_left['truck']}")
    print(f"  Motorcycles: {detector.vehicle_counts_left['motorcycle']}")
    print(f"  Bicycles: {detector.vehicle_counts_left['bicycle']}")
    
    print(f"\nRIGHT LANE: {total_right} vehicles")
    print(f"  Cars: {detector.vehicle_counts_right['car']}")
    print(f"  Trucks: {detector.vehicle_counts_right['truck']}")
    print(f"  Motorcycles: {detector.vehicle_counts_right['motorcycle']}")
    print(f"  Bicycles: {detector.vehicle_counts_right['bicycle']}")
    
    print(f"\nTOTAL: {total_left + total_right} vehicles")
    print(f"Processed {frame_count} frames in {frame_count/np.mean(fps_list):.2f}s")
    print(f"Average FPS: {np.mean(fps_list):.2f}")
    print(f"Output saved: {output_path}")
    print("="*60)

print("Processing function defined!")

Processing function defined!


### Step 6: Initialize and Run

In [28]:
# Initialize detector
detector = VehicleDetector(CONFIG)
print(f"\nTarget Classes: {CONFIG['target_classes']}")
print(f"Model saves to: {CONFIG['model_path']}")
print(f"Dual-lane counting mode ACTIVE")

Loading YOLOv8n model: yolov8m.pt
Model loaded successfully!

Target Classes: ['motorcycle', 'bicycle', 'car', 'truck']
Model saves to: ..\models\yolov8m.pt
Dual-lane counting mode ACTIVE


### Step 7: Process Video

In [ ]:
# Run the pipeline
process_video(
    detector,
    CONFIG['video_input'],
    CONFIG['video_output']
)


Processing video: ..\data\video\raw\traffic_video_demo.mp4
Resolution: 1920x1080, FPS: 30.0, Frames: 1050
Processing frames...
Progress: 1/1050 (0.1%)
Progress: 30/1050 (2.9%)
Progress: 60/1050 (5.7%)


### Step 8: Verify Output

In [24]:
# Verify output
output_path = Path(CONFIG['video_output'])

if output_path.exists():
    file_size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"\n✓ Output video created!")
    print(f"  Location: {output_path}")
    print(f"  Size: {file_size_mb:.2f} MB")
    
    cap = cv2.VideoCapture(str(output_path))
    if cap.isOpened():
        print(f"  Frames: {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}")
        print(f"  FPS: {cap.get(cv2.CAP_PROP_FPS):.2f}")
        cap.release()
        print("✓ Video is valid!")
else:
    print(f"✗ ERROR: Output not found at {output_path}")


✓ Output video created!
  Location: ..\data\video\processed\output_lane_count.mp4
  Size: 101.07 MB
  Frames: 1022
  FPS: 30.00
✓ Video is valid!
